# SARIMA Forecasting Experiment

## 1. Environment Setup and Imports

In [1]:
import sys
import time
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import mlflow

from sklearn.metrics import mean_absolute_error, root_mean_squared_error

project_root = Path.cwd()

if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))


from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tools.sm_exceptions import ConvergenceWarning


warnings.filterwarnings(
    "ignore",
    category=ConvergenceWarning
)

warnings.filterwarnings(
    "ignore",
    category=UserWarning
)


from src.walmart_forecasting.data import load_processed_data

from src.walmart_forecasting.paths import TABLES_DIR

from src.walmart_forecasting.validation import (
    chronological_holdout,
    expanding_window_splits,
)

from src.walmart_forecasting.experiment import (
    HOLIDAY_WEIGHT,
    NON_HOLIDAY_WEIGHT,
    CV_FOLDS,
    CV_VALIDATION_WEEKS,
    FINAL_HOLDOUT_WEEKS,
    DEFAULT_RANDOM_SEED,
    build_common_parameters,
    build_result_row,
    save_architecture_result,
    make_run_name,
)

from src.walmart_forecasting.tracking import mlflow_run

from src.walmart_forecasting.metrics import weighted_mae


np.random.seed(DEFAULT_RANDOM_SEED)

print("Setup complete.")

Setup complete.


## 2. Experiment Configuration

Defines:
- model architecture
- evaluation scope
- feature set
- SARIMA seasonality
- MLflow experiment settings

In [2]:
ARCHITECTURE = "sarima"
STAGE = "tuning"
FEATURE_SET = "target_holiday_v1"
PREPROCESSING = "complete_series_only_v1"
FORECAST_STRATEGY = "local_per_series"
EVALUATION_SCOPE = "representative_series"
TRIAL_NAME = "sarima_representative_v1"
EXPERIMENT_NAME = "SARIMA_Training"
SEASONAL_PERIOD = 52
N_SARIMA_SERIES = 30
EXOG_COLUMNS = [
    "IsHoliday"
]
RESULT_FILE_NAME = (
    "sarima_representative_results.csv"
)
print("Configuration loaded.")

Configuration loaded.


## 3. Load Processed Walmart Data

Load the prepared training dataset and ensure chronological ordering by:
- Store
- Department
- Date

In [3]:
processed = load_processed_data()

merged_train = processed.train.copy()

merged_train["Date"] = pd.to_datetime(
    merged_train["Date"]
)


merged_train = (
    merged_train
    .sort_values(
        [
            "Store",
            "Dept",
            "Date"
        ]
    )
    .reset_index(drop=True)
)


print(
    f"Train rows: {len(merged_train):,}"
)

merged_train.head()

Train rows: 421,570


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315
2,1,1,2010-02-19,41595.55,False,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315
3,1,1,2010-02-26,19403.54,False,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315
4,1,1,2010-03-05,21827.90,False,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315


## 4. Select Complete Time Series

Remove incomplete Store-Department series to avoid target leakage.

Only series containing the full weekly history are used for SARIMA evaluation.

In [4]:
series_summary = (
    merged_train
    .groupby(
        [
            "Store",
            "Dept"
        ]
    )
    .agg(
        first_date=("Date", "min"),
        last_date=("Date", "max"),
        n_weeks=("Date", "nunique"),
    )
    .reset_index()
)


complete_series = series_summary[
    (
        series_summary["first_date"]
        ==
        merged_train["Date"].min()
    )
    &
    (
        series_summary["last_date"]
        ==
        merged_train["Date"].max()
    )
    &
    (
        series_summary["n_weeks"]
        ==
        merged_train["Date"].nunique()
    )
]


print(
    f"Complete series: {len(complete_series)}"
)

Complete series: 2660


## 5. Create Representative Evaluation Sample

Select representative Store-Department series:

- low-volume series
- medium-volume series
- high-volume series

The same sample should be reused across ARIMA, SARIMA, and Seasonal Naive comparisons.

In [5]:
def build_weekly_series(
    dataframe,
    store,
    dept,
):

    series = (
        dataframe[
            (dataframe["Store"] == store)
            &
            (dataframe["Dept"] == dept)
        ]
        .sort_values("Date")
        .set_index("Date")
        .asfreq("W-FRI")
    )


    if series["Weekly_Sales"].isna().any():
        raise ValueError(
            f"Store {store} Dept {dept} is incomplete"
        )


    series["Store"] = store
    series["Dept"] = dept


    if "IsHoliday" in series.columns:

        series["IsHoliday"] = (
            series["IsHoliday"]
            .fillna(0)
            .astype(float)
        )


    return series.reset_index()

## 6. Build Weekly Forecasting Series

Prepare individual Store-Department time series.

No target interpolation is performed to prevent validation leakage.

In [6]:
volume_summary = (
    merged_train
    .groupby(
        [
            "Store",
            "Dept"
        ]
    )
    ["Weekly_Sales"]
    .mean()
    .reset_index()
)


volume_summary = volume_summary.merge(
    complete_series[
        [
            "Store",
            "Dept"
        ]
    ],
    on=[
        "Store",
        "Dept"
    ],
    how="inner",
)


volume_summary = (
    volume_summary
    .sort_values("Weekly_Sales")
    .reset_index(drop=True)
)


low_volume = volume_summary.head(10)


high_volume = volume_summary.tail(10)


middle_start = (
    len(volume_summary) // 2
    -
    5
)

medium_volume = volume_summary.iloc[
    middle_start:middle_start + 10
]


selected = pd.concat(
    [
        low_volume,
        medium_volume,
        high_volume,
    ]
)


selected_keys = list(
    zip(
        selected["Store"],
        selected["Dept"],
    )
)


print(
    f"Selected series: {len(selected_keys)}"
)

print(selected)

Selected series: 30
      Store  Dept   Weekly_Sales
0        38    52      21.399580
1        44    52      25.698462
2        37    52      32.246503
3        42    52      33.125734
4        44    25      34.191888
5        31    19      41.546503
6        38    25      52.630699
7        42    28      57.608462
8        38    12      63.583706
9        30    42      64.555175
1325     12    32    9506.941888
1326     45     3    9508.014965
1327      4    42    9511.818531
1328     14    33    9513.732028
1329      7     1    9542.801259
1330     11    98    9570.351469
1331     43    98    9594.867483
1332     29    14    9601.125245
1333     11    20    9634.961958
1334     27    24    9648.699161
2650      2    95  143588.751888
2651     14    95  144446.932517
2652     27    92  146518.141399
2653      4    95  147236.473706
2654     20    95  150613.955385
2655      4    92  159365.107902
2656     13    92  162034.099301
2657     20    92  164633.741538
2658      2    92  1648

## 7. Evaluation Metrics

Define competition-style metrics:

- Weighted MAE (WMAE)
- MAE
- RMSE

Metrics are calculated over all validation predictions together.

In [7]:
ARIMA_GRID = [
    (p, d, q)
    for p in [0, 1, 2]
    for d in [0, 1]
    for q in [0, 1, 2]
]


SEASONAL_GRID = [
    (P, D, Q)
    for P in [0, 1]
    for D in [0, 1]
    for Q in [0, 1]
]


print(
    "ARIMA combinations:",
    len(ARIMA_GRID)
)

print(
    "Seasonal combinations:",
    len(SEASONAL_GRID)
)

ARIMA combinations: 18
Seasonal combinations: 8


## 8. SARIMA Model Selection

Search for:

- ARIMA order (p,d,q)
- Seasonal order (P,D,Q,52)

Selection is performed using only training data inside each CV fold.

In [8]:
def select_arima_order(
    train_values,
    train_exog=None,
):

    best_order = None
    best_aicc = np.inf


    for order in ARIMA_GRID:

        try:

            model = SARIMAX(
                train_values,
                exog=train_exog,
                order=order,
                enforce_stationarity=False,
                enforce_invertibility=False,
            ).fit(
                disp=False
            )


            if model.aicc < best_aicc:
                best_aicc = model.aicc
                best_order = order


        except Exception:
            continue


    return best_order, best_aicc

In [9]:
def select_sarima_model(
    train_values,
    order,
    train_exog=None,
):

    best_model = None
    best_seasonal = None
    best_aicc = np.inf


    for P, D, Q in SEASONAL_GRID:

        try:

            model = SARIMAX(
                train_values,
                exog=train_exog,
                order=order,
                seasonal_order=(
                    P,
                    D,
                    Q,
                    SEASONAL_PERIOD,
                ),
                enforce_stationarity=False,
                enforce_invertibility=False,
            ).fit(
                disp=False
            )


            if model.aicc < best_aicc:
                best_aicc = model.aicc
                best_model = model
                best_seasonal = (
                    P,
                    D,
                    Q,
                )


        except Exception:
            continue


    return (
        best_model,
        best_seasonal,
        best_aicc,
    )

In [10]:
run_name = make_run_name(
    architecture=ARCHITECTURE,
    stage=STAGE,
    feature_set=FEATURE_SET,
    trial_name=TRIAL_NAME,
)


common_params = build_common_parameters(
    architecture=ARCHITECTURE,
    stage=STAGE,
    feature_set=FEATURE_SET,
    preprocessing=PREPROCESSING,
    evaluation_scope=EVALUATION_SCOPE,
    forecast_strategy=FORECAST_STRATEGY,
    series_count=len(selected_keys),
    random_seed=DEFAULT_RANDOM_SEED,
    extra_parameters={
        "seasonal_period": SEASONAL_PERIOD,
        "seasonal_grid_size": len(SEASONAL_GRID),
        "arima_grid_size": len(ARIMA_GRID),
        "exog_columns": EXOG_COLUMNS,
        "order_selection": "per_fold_aicc",
    },
)


TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


results_path = (
    TABLES_DIR
    /
    RESULT_FILE_NAME
)

## 10. SARIMA Training and Evaluation

Run:

- expanding-window cross-validation
- final chronological holdout evaluation
- prediction collection
- metric calculation
- MLflow logging

The MLflow run covers the complete training duration.

In [13]:
prediction_rows = []

per_series_rows = []

cv_fold_scores = defaultdict(list)

total_fit_seconds = 0.0
total_predict_seconds = 0.0


with mlflow_run(
    experiment_name=EXPERIMENT_NAME,
    run_name=run_name,
    parameters=common_params,
    tags={
        "architecture": ARCHITECTURE,
        "stage": STAGE,
        "scope": EVALUATION_SCOPE,
    },
):

    experiment_start = time.perf_counter()


    for i, (store, dept) in enumerate(selected_keys):

        print(
            f"\n[{i+1}/{len(selected_keys)}] "
            f"Store {store} Dept {dept}"
        )


        try:

            series_df = build_weekly_series(
                merged_train,
                store,
                dept,
            )


            holdout_split = chronological_holdout(
                series_df,
                validation_weeks=FINAL_HOLDOUT_WEEKS,
            )


            cv_splits = expanding_window_splits(
                holdout_split.train,
                n_splits=CV_FOLDS,
                validation_weeks=CV_VALIDATION_WEEKS,
            )


        except ValueError as exc:

            print(
                f"Skipping: {exc}"
            )

            continue



        series_cv_scores = []



        # ==========================
        # CV
        # ==========================

        for fold_index, fold in enumerate(cv_splits):

            y_train = fold.train["Weekly_Sales"].values
            y_valid = fold.validation["Weekly_Sales"].values


            train_exog = (
                fold.train[EXOG_COLUMNS]
                .values
            )

            valid_exog = (
                fold.validation[EXOG_COLUMNS]
                .values
            )


            fit_start = time.perf_counter()


            order, _ = select_arima_order(
                y_train,
                train_exog,
            )


            if order is None:
                continue


            model, seasonal_order, _ = (
                select_sarima_model(
                    y_train,
                    order,
                    train_exog,
                )
            )


            total_fit_seconds += (
                time.perf_counter()
                -
                fit_start
            )


            if model is None:
                continue



            predict_start = time.perf_counter()


            forecast = model.forecast(
                steps=len(y_valid),
                exog=valid_exog,
            )


            total_predict_seconds += (
                time.perf_counter()
                -
                predict_start
            )


            fold_wmae = weighted_mae(
                y_valid,
                forecast,
                fold.validation["IsHoliday"].values,
            )


            series_cv_scores.append(
                fold_wmae
            )


            cv_fold_scores[fold_index].append(
                fold_wmae
            )



        if len(series_cv_scores) == 0:
            continue



        # ==========================
        # HOLDOUT
        # ==========================

        y_train = holdout_split.train["Weekly_Sales"].values

        y_valid = holdout_split.validation["Weekly_Sales"].values


        train_exog = (
            holdout_split.train[EXOG_COLUMNS]
            .values
        )

        valid_exog = (
            holdout_split.validation[EXOG_COLUMNS]
            .values
        )


        fit_start = time.perf_counter()


        order, _ = select_arima_order(
            y_train,
            train_exog,
        )


        if order is None:
            continue


        model, seasonal_order, _ = (
            select_sarima_model(
                y_train,
                order,
                train_exog,
            )
        )


        total_fit_seconds += (
            time.perf_counter()
            -
            fit_start
        )


        if model is None:
            continue



        predict_start = time.perf_counter()


        forecast = model.forecast(
            steps=len(y_valid),
            exog=valid_exog,
        )


        total_predict_seconds += (
            time.perf_counter()
            -
            predict_start
        )



        prediction_rows.append(
            pd.DataFrame(
                {
                    "Store": store,
                    "Dept": dept,
                    "Date": (
                        holdout_split.validation["Date"]
                    ),
                    "actual": y_valid,
                    "prediction": forecast,
                    "IsHoliday": (
                        holdout_split.validation["IsHoliday"]
                        .values
                    ),
                }
            )
        )



        per_series_rows.append(
            {
                "Store": store,
                "Dept": dept,
                "order": order,
                "seasonal_order": seasonal_order,
                "cv_wmae_mean": float(
                    np.mean(series_cv_scores)
                ),
                "cv_wmae_std": float(
                    np.std(series_cv_scores)
                ),
            }
        )


        print(
            f"order={order}, seasonal={seasonal_order}"
        )



    # ==========================
    # GLOBAL METRICS
    # ==========================

    all_predictions = pd.concat(
        prediction_rows,
        ignore_index=True,
    )


    holdout_wmae = weighted_mae(
        all_predictions["actual"],
        all_predictions["prediction"],
        all_predictions["IsHoliday"],
    )


    holdout_mae = mean_absolute_error(
        all_predictions["actual"],
        all_predictions["prediction"],
    )


    holdout_rmse = root_mean_squared_error(
        all_predictions["actual"],
        all_predictions["prediction"],
    )


    cv_fold_level_scores = [
        np.mean(values)
        for values in cv_fold_scores.values()
    ]


    overall_metrics = {
        "cv_wmae_mean": float(
            np.mean(cv_fold_level_scores)
        ),
        "cv_wmae_std": float(
            np.std(cv_fold_level_scores)
        ),
        "holdout_wmae": float(
            holdout_wmae
        ),
        "holdout_mae": float(
            holdout_mae
        ),
        "holdout_rmse": float(
            holdout_rmse
        ),
    }



    results_df = pd.DataFrame(
        per_series_rows
    )


    results_df.to_csv(
        results_path,
        index=False,
    )


    # ==========================
    # MLflow logging happens BEFORE run closes
    # ==========================

    mlflow.log_metrics(
        {
            **overall_metrics,
            "fit_seconds": total_fit_seconds,
            "predict_seconds": total_predict_seconds,
        }
    )


    mlflow.log_artifact(
        str(results_path)
    )


    total_runtime = (
        time.perf_counter()
        -
        experiment_start
    )


    mlflow.log_metric(
        "total_runtime_seconds",
        total_runtime,
    )



print("\nFinished SARIMA experiment")

for key, value in overall_metrics.items():
    print(
        f"{key}: {value:.3f}"
    )

Initialized MLflow to track repo "lkhiz23/walmart-store-sales-forecasting"

Repository lkhiz23/walmart-store-sales-forecasting initialized!

[1/30] Store 38 Dept 52
[2/30] Store 44 Dept 52
[3/30] Store 37 Dept 52
[4/30] Store 42 Dept 52
[5/30] Store 44 Dept 25
[6/30] Store 31 Dept 19
[7/30] Store 38 Dept 25
[8/30] Store 42 Dept 28
[9/30] Store 38 Dept 12
[10/30] Store 30 Dept 42
[11/30] Store 12 Dept 32
[12/30] Store 45 Dept 3
[13/30] Store 4 Dept 42
[14/30] Store 14 Dept 33
[15/30] Store 7 Dept 1
[16/30] Store 11 Dept 98
[17/30] Store 43 Dept 98
[18/30] Store 29 Dept 14
[19/30] Store 11 Dept 20
[20/30] Store 27 Dept 24
[21/30] Store 2 Dept 95
[22/30] Store 14 Dept 95
[23/30] Store 27 Dept 92
[24/30] Store 4 Dept 95
[25/30] Store 20 Dept 95
[26/30] Store 4 Dept 92
[27/30] Store 13 Dept 92
[28/30] Store 20 Dept 92
[29/30] Store 2 Dept 92
[30/30] Store 14 Dept 92
🏃 View run sarima__tuning__target_only_v1__large_grid_no_exog_v1__s42 at: https://dagshub.com/lkhiz23/walmart-store-sales-forecasting.mlflow/#/experiments/3/runs/dca7bfa7f8ae4acda16812c565419285
🧪 View experiment at: https://dagshub.com/lkhiz23/walmart-store-sales-fo